# Tag → FacetType 'Experience' migration

Tracker row **#2 (Tag)** — legacy Strapi `tags` become **facet values** under a
single new **FacetType 'Experience'**.

Scope decisions (2026-08-05, confirmed in tracker):
- All tags land under ONE facet type named **Experience** (`experience`),
  `applies_to_collection_type_id = NULL` (applies broadly),
  `allows_multiple = TRUE` (a postcard has many experiences).
- **Owned by Postcard only** — FacetAssignments use `owned_type = 'postcard'`.
  Property-level filtering rolls up from child postcards at query time; there
  is **no direct Collection-level assignment** (unlike Theme).
- **FacetAssignments are NOT created here** — postcards aren't migrated yet
  (tracker #16 depends on Album + Tag). This notebook saves
  `legacy_tag_id_map.json` (legacy tag id → facet_value id); the postcard
  migration creates the assignments from each postcard's `tags` relation.
- Duplicate tag names (8 pairs, e.g. `buddhist temple` ×2) are **merged**:
  both legacy ids map to the same facet_value — no `-2` suffix junk values.
- `tag_group` has no home in the facet schema (no grouping level between
  FacetType and FacetValue). The linkage is preserved to
  `legacy_tag_groups.json` for tracker row **#29 (Tag-group)** to decide on
  later — nothing is lost, but it is not written to the DB here.
- Dropped: `follow_tags` (blocked Circle work, tracker #26), `createdAt` /
  `updatedAt` (Strapi housekeeping).
- The new `tags` TABLE is a different thing (curated postcard feature tags +
  persona tags, seeded by `seed.py`) — legacy tags do **not** go there.

Prerequisites: schema deployed + `python scripts/seed.py` run. Independent of
geo/media/company/users — only needs the DB.

Run cells top to bottom. Idempotent — safe to re-run.

In [1]:
import os, re, json
from pathlib import Path

import requests
import psycopg
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", (text or "").lower()).strip("-") or None


def attrs(item):
    """Entry fields — Strapi v4 nests them under 'attributes', v5 is flat."""
    return item.get("attributes", item)


def rel(obj):
    """Unwrap a populated relation — v4: {'data': {'attributes': {...}}}, v5: flat dict."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    if not obj:
        return None
    return obj.get("attributes", obj)


def fetch_all(path, params=None):
    """Fetch every page of a Strapi collection endpoint (data/meta envelope)."""
    items, page = [], 1
    while True:
        p = {"pagination[page]": page, "pagination[pageSize]": 100, "sort": "id", **(params or {})}
        r = requests.get(f"{CMS_BASE_URL}{path}", headers=HEADERS, params=p, timeout=120)
        r.raise_for_status()
        body = r.json()
        items.extend(body["data"])
        pg = body.get("meta", {}).get("pagination", {})
        if page >= pg.get("pageCount", 1):
            return items
        page += 1


conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])

connected to: development


## 1. Fetch all legacy tags (~8 paginated requests, expect 730)

`populate=tag_group` so the group linkage can be preserved to the side file.
Sorted by id so the merge of duplicate names is stable across re-runs
(lowest legacy id wins).

In [2]:
tags = sorted(fetch_all("/api/tags", {"populate": "tag_group"}), key=lambda t: t["id"])
print(f"fetched {len(tags)} tags")

groups = fetch_all("/api/tag-groups")
print(f"fetched {len(groups)} tag_groups:", [attrs(g).get("name") for g in groups])

fetched 701 tags
fetched 10 tag_groups: ['Architecture & Design', 'Art, Craft & Textile', 'Culture & Community', 'Food & Drink', 'History & Heritage', 'Health & Wellness', 'Nature & Wildlife', 'Sports & Adventure', 'Spirituality & Religion', 'Romance & Celebrations']


## 2. FacetType 'Experience'

One row, upserted on slug. Distinct from the seeded **Experience Theme**
(`experience-theme`) — Theme is assigned directly at Collection level,
Experience only at Postcard level.

In [3]:
conn.rollback()  # clear any aborted transaction from a previous failed run

with conn.cursor() as cur:
    cur.execute(
        """
        INSERT INTO facet_types (name, slug, applies_to_collection_type_id, allows_multiple)
        VALUES ('Experience', 'experience', NULL, TRUE)
        ON CONFLICT (slug) DO UPDATE
        SET name = EXCLUDED.name,
            applies_to_collection_type_id = EXCLUDED.applies_to_collection_type_id,
            allows_multiple = EXCLUDED.allows_multiple
        RETURNING id
        """
    )
    EXPERIENCE_FT_ID = cur.fetchone()[0]
conn.commit()
print("facet_type 'Experience' id:", EXPERIENCE_FT_ID)

facet_type 'Experience' id: 1


## 3. Tag → `facet_values`

- slug generated from name (legacy has none); name is stored trimmed but
  otherwise as-is (lowercase in legacy, e.g. `desert dining`).
- duplicate slugs merge into ONE facet_value — every legacy id still gets a
  map entry, pointing at the shared row.
- upsert on `(facet_type_id, slug)`.

In [4]:
conn.rollback()

tag_to_facet_value = {}   # legacy tag id -> facet_value id
fv_id_by_slug = {}        # slug -> facet_value id (dedupe within the run)
merged, skipped_no_name = [], []

with conn.cursor() as cur:
    for t in tags:
        a = attrs(t)
        name = (a.get("name") or "").strip()
        if not name:
            skipped_no_name.append(t["id"])
            continue
        slug = slugify(name)

        if slug in fv_id_by_slug:  # duplicate tag name -> merge onto existing value
            tag_to_facet_value[t["id"]] = fv_id_by_slug[slug]
            merged.append((t["id"], name))
            continue

        cur.execute(
            """
            INSERT INTO facet_values (facet_type_id, name, slug)
            VALUES (%s, %s, %s)
            ON CONFLICT (facet_type_id, slug) DO UPDATE
            SET name = EXCLUDED.name
            RETURNING id
            """,
            (EXPERIENCE_FT_ID, name, slug),
        )
        fv_id = cur.fetchone()[0]
        fv_id_by_slug[slug] = fv_id
        tag_to_facet_value[t["id"]] = fv_id

conn.commit()
print(f"facet_values upserted: {len(fv_id_by_slug)}")
print(f"legacy tags mapped   : {len(tag_to_facet_value)}")
print(f"merged duplicates ({len(merged)}): {merged}")   # expect 8
print(f"skipped (no name): {skipped_no_name}")           # expect []

facet_values upserted: 676
legacy tags mapped   : 701
merged duplicates (25): [(10219, 'Buddhist temple'), (10229, 'tuk tuk ride'), (10231, 'adventure park visit'), (10299, 'helicopter ride'), (10300, 'balinese culture'), (10312, 'valley visit'), (10405, 'gandola ride'), (10591, 'cricket bat factory visit'), (10592, 'camping'), (10593, 'hippo spotting'), (10594, 'atv ride'), (10595, 'charles darwin research station'), (10596, 'double-humped bactrian camel visit'), (10601, 'Local Food'), (10607, 'Temple Visit'), (10609, 'Mountain Landscape'), (10613, 'Hiking'), (10616, 'Monastery Visit'), (10618, 'Wellness Experience'), (10619, 'Local Culture'), (10620, 'Cooking Class'), (10623, 'apple orchard'), (10624, 'baking'), (10625, 'Heritage Walk'), (10626, 'Fine Dining')]
skipped (no name): []


## 4. Save the map files

- `legacy_tag_id_map.json` — legacy tag id → facet_value id. The **postcard
  migration needs it** to create `facet_assignments`
  (`owned_type='postcard'`) from each postcard's `tags` relation. Rename per
  environment like the user/album maps.
- `legacy_tag_groups.json` — tag-group definitions + per-tag group membership,
  preserved for tracker **#29 (Tag-group)**; not written to the DB here.

In [5]:
out = ROOT / "legacy_tag_id_map.json"
out.write_text(json.dumps({str(k): str(v) for k, v in tag_to_facet_value.items()}, indent=2))
print(f"saved {len(tag_to_facet_value)} legacy->new tag id mappings to {out}")

groups_out = ROOT / "legacy_tag_groups_dev.json"
groups_out.write_text(json.dumps({
    "tag_groups": [
        {"id": g["id"], "name": attrs(g).get("name"), "priority": attrs(g).get("priority")}
        for g in groups
    ],
    "tag_to_group": {
        str(t["id"]): (rel(attrs(t).get("tag_group")) or {}).get("name")
        for t in tags
    },
}, indent=2))
print(f"saved tag-group linkage for {len(tags)} tags to {groups_out}")

saved 701 legacy->new tag id mappings to c:\Users\ReTechie\Desktop\postcard\postcard-migration\legacy_tag_id_map.json
saved tag-group linkage for 701 tags to c:\Users\ReTechie\Desktop\postcard\postcard-migration\legacy_tag_groups_dev.json


## 5. Verification

Expected: 730 legacy tags → **722 facet_values** (8 duplicate names merged),
730 map entries, facet type `experience` with `allows_multiple = TRUE`,
0 assignments (created later by the postcard migration), 0 duplicate slugs.

In [6]:
with conn.cursor() as cur:
    cur.execute("SELECT id, name, slug, applies_to_collection_type_id, allows_multiple FROM facet_types ORDER BY id")
    for row in cur.fetchall():
        print("facet_type:", row)
    for label, q in [
        ("experience values",   "SELECT COUNT(*) FROM facet_values WHERE facet_type_id = %s"),
        ("assignments (want 0)", "SELECT COUNT(*) FROM facet_assignments WHERE facet_value_id IN (SELECT id FROM facet_values WHERE facet_type_id = %s)"),
        ("dup slugs (want 0)",  "SELECT COUNT(*) FROM (SELECT slug FROM facet_values WHERE facet_type_id = %s GROUP BY slug HAVING COUNT(*) > 1) d"),
    ]:
        cur.execute(q, (EXPERIENCE_FT_ID,))
        print(f"{label:20}: {cur.fetchone()[0]}")
    cur.execute("SELECT name FROM facet_values WHERE facet_type_id = %s ORDER BY name LIMIT 10", (EXPERIENCE_FT_ID,))
    print("sample values:", [r[0] for r in cur.fetchall()])
conn.close()

facet_type: (1, 'Experience', 'experience', None, True)
experience values   : 676
assignments (want 0): 0
dup slugs (want 0)  : 0
sample values: ['Adventure Park Visit', 'african wilderness', 'air rifle shooting', 'al fresco dining', 'all-female ranger meet', 'aloe vera plantation visit', 'andean astronomy', 'andean music', 'anglo-indian cuisine', 'animal conservation']
